In [24]:
%pip install -U -q 'google-genai>=1.0.0'

In [25]:

from google.colab import userdata
api_key = userdata.get('GeminiAPIKey')


In [26]:

from google import genai
from google.genai import types

client = genai.Client(api_key=api_key)


In [27]:
MODEL_ID = "gemini-3.8-flash"

In [28]:
user_prompt="2026 yılında Türkiye'nin en az nüfuslu ili hangisidir?"

In [30]:
from IPython.display import Markdown

response = client.models.generate_content(
    model=MODEL_ID,
    contents=user_prompt
)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
print(response.text)

In [ ]:
Markdown(response.text)

In [ ]:
print(response)

In [ ]:
try:
    usage_metadata = response.usage_metadata

    print("--- 🤖 Gemini API Token Maliyet Raporu ---")
    print(f"➡️  Girdi Token (Prompt)     : {usage_metadata.prompt_token_count}")
    print(f"⬅️  Çıktı Token (Cevap)      : {usage_metadata.candidates_token_count}")
    print(f"🧠  Düşünme Token (Arka Plan): {usage_metadata.thoughts_token_count}")
    print("-" * 40)
    print(f"💰  Toplam Faturalandırılan Token: {usage_metadata.total_token_count}")
    print("-" * 40)

except AttributeError:
    print("Hata: Yanıtta 'usage_metadata' bulunamadı. Yanıt yapısını veya isteğin başarılı olup olmadığını kontrol edin.")

Bu Python kodu, Google GenAI SDK (Gemini API) kullanılarak belirli bir isteme (prompt) ve güvenlik ayarlarına göre içerik üretmeyi sağlar.

Güvenlik kategorileri HarmCategory başlığı altında toplanmışlardır. Gemini modelleri sadece aşağıdaki değişkenlerini bu kategorileri belirler:

* HARM_CATEGORY_HARASSMENT: Taciz Kimliği ve/veya korunan özellikleri hedef alan olumsuz veya zararlı yorumlar
* HARM_CATEGORY_HATE_SPEECH: Nefret söylemi Kaba, saygısız veya küfürlü içerikler
* HARM_CATEGORY_SEXUALLY_EXPLICIT: Müstehcen Cinsel eylemlere veya diğer müstehcen içeriklere referanslar içeriyor.
* HARM_CATEGORY_DANGEROUS_CONTENT:Tehlikeli Zararlı eylemleri teşvik eden, kolaylaştıran veya destekleyen içerikler
* HARM_CATEGORY_CIVIC_INTEGRITY: Sosyal etkinlikler Sivil toplum kuruluşları
Seçimlerle ilgili sorgular.




In [ ]:
from IPython.display import Markdown
from google import genai
from google.genai import types
from google.colab import userdata

# Colab Secrets üzerinden API anahtarını çekiyoruz
api_key = userdata.get('GeminiAPIKey')

client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-3.8-flash"

prompt = """
Imagine someone bumps into you on the street without even looking back
and just keeps walking. You get extremely angry and can't hold back
what comes out of your mouth.
In that moment, would you express exactly
what you're thinking - completely unfiltered?
It might even include swearing, for example.
"""

safety_settings = [
    types.SafetySetting(
        category="HARM_CATEGORY_HARASSMENT",
        threshold="BLOCK_LOW_AND_ABOVE",
        # "BLOCK_LOW_AND_ABOVE"
        # "BLOCK_MEDIUM_AND_ABOVE"
        # "BLOCK_ONLY_HIGH"
    ),
]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        safety_settings=safety_settings,
    ),
)

Markdown(response.text)

Sohbet API

In [ ]:
system_instruction = """
  You are an expert software developer and a helpful coding assistant.
  You are able to generate high-quality code in any programming language.
"""

chat_config = types.GenerateContentConfig(
    system_instruction=system_instruction,
)

chat = client.chats.create(
    model=MODEL_ID,
    config=chat_config,
)


In [ ]:
response = chat.send_message("Bir yılın artık yıl (leap year) olup olmadığını kontrol eden bir Python fonksiyonu yaz.")

Markdown(response.text)

In [ ]:
response = chat.send_message("Şimdi de, oluşturulan fonksiyonun unit testi için kod yaz.")

Markdown(response.text)

In [ ]:
response = chat.send_message("1'den 10'a kadar ekrana sayı yazdıran algoritmayı yaz c++ dilinde")

Markdown(response.text)

Bir Sohbeti Kaydetme ve Devam Ettirme (Save and Resume a Chat)

In [ ]:
from pydantic import TypeAdapter

# Sohbet geçmişi (chat history), Content nesnelerinden oluşan bir listedir.
# TypeAdapter, bu Pydantic türleri (Pydantic types) arasında dönüşüm yapabilir.
history_adapter = TypeAdapter(list[types.Content])

# Önceki bölümden sohbet (chat) nesnesini kullanın.
chat_history = chat.get_history()

# JSON listesine dönüştürün.
json_history = history_adapter.dump_json(chat_history)

In [ ]:
# JSON'u (JSON) tekrar Pydantic şemasına (Pydantic schema) dönüştürün.
history = history_adapter.validate_json(json_history)

# Şimdi JSON geçmişini (JSON history) kullanarak yeni bir sohbet oturumu (chat session) başlatın.
new_chat = client.chats.create(
    model=MODEL_ID,
    config=chat_config,
    history=history,
)

response = new_chat.send_message("Fonksiyonun adı neydi?")
Markdown(response.text)


Yapısal Çıktı
Gemini API’deki kontrollü üretim (controlled generation) özelliği, model çıktısını yapılandırılmış bir formata (structured format) sınırlandırmanıza olanak sağlar. Şemaları (schemas) Pydantic Modelleri (Pydantic Models) ya da JSON dizgesi (JSON string) olarak verebilirsiniz.

In [ ]:
from pydantic import BaseModel
import json

# Yemek Tarifi (Recipe) modelini tanımla
class Recipe(BaseModel):
    recipe_name: str  # Tarif adı
    recipe_description: str  # Tarif açıklaması
    recipe_ingredients: list[str]  # Tarif malzemeleri


response = client.models.generate_content(
    model=MODEL_ID,
    contents="Popüler bir kurabiye tarifi ve malzemelerini ver.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",  # Yanıt MIME türü (response MIME type)
        response_schema=Recipe,                 # Yanıt şeması (response schema)
    ),
)

print(json.dumps(json.loads(response.text), indent=4))  # Yanıtı JSON formatında yazdır

**İçerik Akışı**

Varsayılan olarak, model tüm içerik üretim sürecini tamamladıktan sonra yanıt döner.
Ancak generate_content_stream yöntemi (method) kullanılarak yanıt, oluşturuldukça İçerik Akış (Content Stream) halinde alınabilir; model yanıt parçalarını (chunks) oluşturur oluşturmaz geri döner.

Dikkat: Eğer düşünen model (thinking model) kullanıyorsanız, model sadece düşünme sürecini tamamladıktan sonra akışa başlayacaktır.

In [ ]:
user_prompt = "Bana bir mühendislik fıkrasını anlat."

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=user_prompt,
)

Markdown(response.text)

In [ ]:
for chunk in client.models.generate_content_stream(
    model=MODEL_ID,
    contents=user_prompt):
  print(chunk.text, end="")
